<img src="https://github.com/pavlek358/eda-exploring-and-joining-data-with-pandas/blob/main/assets/ga-logo.png?raw=1" style="float: left; margin: 20px; height: 55px">

# Descriptive statistics and data joins in `pandas`

---

### About
A guided walkthrough of calculating descriptive statistics with `pandas`, summarising data using `groupby` and joining data.

### Learning Objectives
- Define different types of data.
- Explore descriptive statistics of a data set using `pandas`.
- Produce summary statistics with respect to some categorical variable using `groupby` in `pandas`.
- Explain different kinds of join and merge data sets together using `pandas`.


### Notebook Guide
- Scenario
- Calculating descriptive statistics in `pandas`
    - Re-cap: data types
- Group-level analysis in `pandas` using **split-apply-combine** (`groupby`)
- Merging data sets together with `pandas`

# Scenario

You're an analyst working for CoolBricks - a new toy company who wants to compete with LEGO by producing their own bricks and sets. To better understand what kind of sets to create and at what price points, you are tasked with analyzing LEGO's catalogue over time.

The data comes from Kaggle: [https://www.kaggle.com/datasets/alexracape/lego-sets-and-prices-over-time](https://www.kaggle.com/datasets/alexracape/lego-sets-and-prices-over-time)

We are going to use `pandas` to read and explore this data.

The question we want to answer is: **How do LEGO products vary across different types/themes?**

# Calculating descriptive statistics in `pandas`
---

In [ ]:
import pandas as pd

Let's read in the data and preview it.

In [5]:
import pandas as pd

lego_df = pd.read_csv('https://raw.githubusercontent.com/pavlek358/DSB-Unit-02/refs/heads/main/data/lego.csv')
lego_df.head()

,Set_ID,Name,Year,Theme,Theme_Group,Subtheme,Category,Packaging,Num_Instructions,Availability,Pieces,Minifigures,Owned,Rating,USD_MSRP,Total_Quantity,Current_Price
0,75-1,PreSchool Set,1975,PreSchool,Pre-school,NaN,Normal,{Not specified},0,{Not specified},16.0,NaN,10.0,0.0,NaN,NaN,NaN
1,77-1,PreSchool Set,1975,PreSchool,Pre-school,NaN,Normal,{Not specified},0,{Not specified},20.0,NaN,11.0,0.0,NaN,NaN,NaN
2,077-1,Pre-School Set,1975,Duplo,Pre-school,NaN,Normal,{Not specified},0,{Not specified},21.0,NaN,10.0,0.0,NaN,0.0,NaN
3,78-1,PreSchool Set,1975,PreSchool,Pre-school,NaN,Normal,{Not specified},0,{Not specified},32.0,NaN,8.0,0.0,NaN,NaN,NaN
4,78-3,Basic Set,1975,Samsonite,Vintage,Basic set,Normal,Box,0,{Not specified},330.0,NaN,10.0,0.0,NaN,0.0,NaN


Let's look at the column information

In [11]:
lego_df.columns     # just column names
#lego_df.dtypes      # column names + data types
#lego_df.info()      # detailed column information
#lego_df.shape       # rows and columns

Index(['Set_ID', 'Name', 'Year', 'Theme', 'Theme_Group', 'Subtheme',
       'Category', 'Packaging', 'Num_Instructions', 'Availability', 'Pieces',
       'Minifigures', 'Owned', 'Rating', 'USD_MSRP', 'Total_Quantity',
       'Current_Price'],
      dtype='object')

---

## Recap: data types

We can categorize data into types:

- **Categorical** (discrete categories)
  - Nominal (no inherent ordering, e.g. color)
  - Ordinal (discrete, ordered categories where there is an inherent order e.g., t-shirt sizes S/M/L, or number of visits to the dentist)
- **Continuous** (numeric values along a scale)
  - Interval (zero doesn't mean an absence, e.g. temperature)
  - Ratio (zero means an absence of the value, e.g. height)

## Give an example from the LEGO data of:

- A **nominal** type - see below
- An **ordinal** type - see below
- An **interval** type - see below
- A **ratio** type - see below

Let's get some high-level information about prices

In [13]:
lego_df.groupby("Theme_Group")["Current_Price"].describe()

,count,mean,std,min,25%,50%,75%,max
Theme_Group,,,,,,,,
Action/Adventure,661.0,87.133444,149.714030,1.0000,17.95200,40.00000,94.491000,1500.0000
Art and crafts,59.0,30.882232,49.520895,0.7300,2.98500,5.00000,27.490000,199.0000
Basic,108.0,22.600684,31.040712,0.0500,6.00000,10.00000,25.000000,200.0000
Constraction,174.0,80.970243,85.342789,4.3543,30.00000,49.25815,99.263125,598.5000
Educational,34.0,133.753232,171.590255,1.4749,23.74000,61.99500,193.742500,795.0000
Girls,25.0,57.785984,72.206312,1.5000,9.60000,35.00000,52.000000,260.0000
Historical,194.0,210.434185,349.259916,4.4458,39.99250,80.00000,254.331750,3000.0000
Junior,38.0,32.721408,18.437945,3.5000,17.25000,28.44000,46.617500,80.0000
Licensed,1633.0,112.382881,256.838080,1.5000,20.79200,47.50000,112.053800,5499.9900


We can also perform multiple aggregations at once

Or use `describe` to get an overview of *all* numeric data

In our research question, we want to know about different themes.

The `Theme` and `Theme_Group` columns look interesting to investigate.

How many different themes are there and what are they?

It looks like themes are contained in theme groups (of which there are fewer).

How many products in each theme group?

In [12]:
lego_df["Theme_Group"].value_counts()

,count
Theme_Group,
Miscellaneous,3407
Modern day,2228
Licensed,2068
Pre-school,1547
Action/Adventure,1177
Basic,827
Model making,725
Technical,594
Constraction,491


This shows us the **mode** of `Theme_Group`.  This is the time that mode is used: for looking at the most common category in categorical data.

Now that we have a sense of what themes exist, we want to know **how prices *vary* across them.**

To do this, we can analyze each group separately using `groupby`, which is a method that performs a "split-apply-combine".

# Group-level analysis in `pandas` using **split-apply-combine** (`groupby`)
---

If we want summary statistics _with respect to some categorical variable_, for example the price of a LEGO set across theme groups, we can use the **Split-Apply-Combine** technique. (This is sometimes called **MapReduce**, but is more of a special case of MapReduce).

* **Split**: Separate your data into different DataFrames, one for each category.
* **Apply**: On each split-up DataFrame, apply some function or transformation (for example, the mean).
* **Combine**: Take the results and combine the split-up DataFrames back into one aggregate DataFrame.

    
<img src="https://github.com/pavlek358/eda-exploring-and-joining-data-with-pandas/blob/main/assets/split_apply_combine.png?raw=1" style="width: 500px;">

_**Source**: Python for Data Analysis, Wes McKinney_

This might sound complicated, but it's actually only two commands in `pandas` (the **Combine** step is done for us).

You can check out a more detailed visual [here](../assets/Split-Apply-Combine-Visual.pdf)!

To answer the question **"What is the average prices of a LEGO set across themes?"** we need to first define the groups.

This is the **Split** step.

In [19]:
lego_df.groupby("Theme_Group")[["USD_MSPR","Current_Price"]].median()

KeyError: "Columns not found: 'USD_MSPR'"

This gives us a `groupby` object on which we need to *apply* some summarization.

When we define the column(s) to summarize (e.g. price) and how to summarize them (e.g. taking the median), `pandas` performs *both* the apply and combine steps.

Finally, we can sort these values in descending order, making this our full command:

# Merging data sets together with `pandas`
---
If we wanted to understand consumer sentiment at the point when LEGO sets were released, we could join some CPI (Consumer Price Index), such as from the table here: https://www.usinflationcalculator.com/inflation/consumer-price-index-and-annual-percent-changes-from-1913-to-2008/

Let's read in a download of this CPI  data that happens to be in JSON (JavaScript Object Notation) format and not a CSV.

In [24]:
cpi_df = pd.read_json("https://raw.githubusercontent.com/pavlek358/DSB-Unit-02/refs/heads/main/data/cpi.json")
cpi_df.head(10)

,Year,avg_cpi
0,1980,80.1
1,1981,88.5
2,1982,94.5
3,1983,97.9
4,1984,102.6
5,1985,106.4
6,1986,108.8
7,1987,112.1
8,1988,116.5
9,1989,122.3


In [23]:
type(cpi)

pandas.core.frame.DataFrame

Every data set you read in to `pandas` becomes a `DataFrame` regardless of its original source.

Even though this data wasn't from a CSV, but a JSON file, it's still a `DataFrame`.

In [28]:
merged_df = lego_df.merge(
    cpi_df,
    left_on= "Year",
    right_on="Year"


)
merged_df.head()

,Set_ID,Name,Year,Theme,Theme_Group,Subtheme,Category,Packaging,Num_Instructions,Availability,Pieces,Minifigures,Owned,Rating,USD_MSRP,Total_Quantity,Current_Price,avg_cpi
0,722-1,"Universal Building Set, 7+",1980,Basic,Basic,NaN,Normal,{Not specified},0,{Not specified},301.0,NaN,1055.0,4.5,NaN,0.0,NaN,80.1
1,733-1,"Universal Building Set, 7+",1980,Basic,Basic,NaN,Normal,{Not specified},0,{Not specified},533.0,NaN,632.0,0.0,NaN,0.0,NaN,80.1
2,744-1,"Universal Building Set with Motor, 7+",1980,Basic,Basic,NaN,Normal,{Not specified},0,{Not specified},537.0,NaN,800.0,4.5,NaN,0.0,NaN,80.1
3,820-1,Red Plates Parts Pack,1980,Basic,Basic,Supplementaries,Normal,{Not specified},0,{Not specified},34.0,NaN,70.0,0.0,NaN,0.0,NaN,80.1
4,822-1,Blue Plates Parts Pack,1980,Basic,Basic,Supplementaries,Normal,{Not specified},0,{Not specified},34.0,NaN,77.0,0.0,NaN,0.0,NaN,80.1


In [29]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14564 entries, 0 to 14563
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Set_ID            14564 non-null  object 
 1   Name              14564 non-null  object 
 2   Year              14564 non-null  int64  
 3   Theme             14564 non-null  object 
 4   Theme_Group       14543 non-null  object 
 5   Subtheme          11336 non-null  object 
 6   Category          14564 non-null  object 
 7   Packaging         14564 non-null  object 
 8   Num_Instructions  14564 non-null  int64  
 9   Availability      14564 non-null  object 
 10  Pieces            12765 non-null  float64
 11  Minifigures       7527 non-null   float64
 12  Owned             14399 non-null  float64
 13  Rating            14564 non-null  float64
 14  USD_MSRP          5837 non-null   float64
 15  Total_Quantity    11916 non-null  float64
 16  Current_Price     5430 non-null   float6

Let's join this data to our LEGO DataFrame.

For this, we can use the `pandas` `merge` method

In [30]:
print("Number of records BEFORE merged:",len(lego_df))
print("Number of records AFTER merged:",len(merged_df))

Number of records BEFORE merged: 14936
Number of records BEFORE merged: 14564


Oops, what happened? We lost some rows!

In [31]:
lego_df["Year"].min()

1975

In [33]:
cpi_df["Year"].min()

1980

In [35]:
lego_df["Year"].unique()

array([1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985,
       1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996,
       1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007,
       2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018,
       2019, 2020, 2021, 2022, 2023])

Ah, our CPI data doesn't start until 1980 but LEGO sets start in 1975!

If we change the merge to a LEFT JOIN, we will keep *all* records in `lego` regardless of whether there's a match in `cpi`

In [37]:
merged_df = lego_df.merge(
    cpi_df,
    left_on= "Year",
    right_on="Year",
    how="left"

)
merged_df.head()

,Set_ID,Name,Year,Theme,Theme_Group,Subtheme,Category,Packaging,Num_Instructions,Availability,Pieces,Minifigures,Owned,Rating,USD_MSRP,Total_Quantity,Current_Price,avg_cpi
0,75-1,PreSchool Set,1975,PreSchool,Pre-school,NaN,Normal,{Not specified},0,{Not specified},16.0,NaN,10.0,0.0,NaN,NaN,NaN,NaN
1,77-1,PreSchool Set,1975,PreSchool,Pre-school,NaN,Normal,{Not specified},0,{Not specified},20.0,NaN,11.0,0.0,NaN,NaN,NaN,NaN
2,077-1,Pre-School Set,1975,Duplo,Pre-school,NaN,Normal,{Not specified},0,{Not specified},21.0,NaN,10.0,0.0,NaN,0.0,NaN,NaN
3,78-1,PreSchool Set,1975,PreSchool,Pre-school,NaN,Normal,{Not specified},0,{Not specified},32.0,NaN,8.0,0.0,NaN,NaN,NaN,NaN
4,78-3,Basic Set,1975,Samsonite,Vintage,Basic set,Normal,Box,0,{Not specified},330.0,NaN,10.0,0.0,NaN,0.0,NaN,NaN


Now we have the right number of records and we can see missing data where there is no CPI information available

In [39]:
print(f"Number of records BEFORE merged: {len(lego_df)}")
print(f"Number of records AFTER merged: {len(merged_df)}")

Number of records BEFORE merged: 14936
Number of records AFTER merged: 14936


# Conclusions and Takeaways

- `pandas` lets us calculate descriptive statistics on individual columns, or multiple columns at once
- `pandas` lets us calculate individual descriptive statistics or multiple aggregations at once
- The `describe` method gives us a high-level numeric summary of our data
- To do statistical analysis on subgroups, we can use `groupby`
- We can join and merge data together from different sources into `pandas` `DataFrames`